In [1]:
# Hitters Contract Prediction — Scikit-Learn Pipeline (Rollback to Last Working, with GroupKFold)

# =====================================================================================
# 1) Setup & Configuration
# =====================================================================================

# --- User configuration ---
DATA_SOURCE_TYPE = "excel"  # 'excel' or 'csv'
HITTERS_FILEPATH = "/Users/bclem213/Desktop/BDA 602/Hitting the Market/Data Files/Updated Cleaned Data/batters_joined_batting.xlsx"
CONTRACTS_FILEPATH = "/Users/bclem213/Desktop/BDA 602/Hitting the Market/Data Files/Updated Cleaned Data/contracts_hitters.xlsx"
STANDARD_SHEET   = "Clean"
CONTRACT_SHEET   = "Clean"

# If using CSVs instead of Excel, set file paths:
# STANDARD_CSV = "/mnt/data/hitters_standard_clean.csv"  # combined hitters
# CONTRACT_CSV = "/mnt/data/contracts_clean.csv"

# Target columns in the contract data
TARGET_YEARS_COL = "contract_years"
TARGET_VALUE_COL = "contract_total_value"

# Keys for joining
KEY_PLAYER = "player_name"
KEY_SEASON = "season"  # ensure this is a string like '2023'
# Optional: if contract aligns to the *first season under new deal*:
KEY_CONTRACT_SEASON = "contract_start_year"  # fallback to season if absent

RANDOM_STATE = 42
N_JOBS = -1



In [2]:
# =====================================================================================
# 2) Imports
# =====================================================================================
import os
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

import warnings
warnings.filterwarnings("ignore")
import joblib



In [3]:
# =====================================================================================
# 3) Load Data
# =====================================================================================
def load_hitters_data():
    """
    Loads hitters from a single Excel sheet (HITTERS_FILEPATH + STANDARD_SHEET)
    and loads contracts from a separate Excel file (CONTRACTS_FILEPATH + CONTRACT_SHEET).
    If DATA_SOURCE_TYPE == 'csv', it falls back to CSV paths (uncomment above if needed).
    """
    if DATA_SOURCE_TYPE.lower() == "excel":
        hitters_df = pd.read_excel(HITTERS_FILEPATH, sheet_name=STANDARD_SHEET)
        con = pd.read_excel(CONTRACTS_FILEPATH, sheet_name=CONTRACT_SHEET)
    else:
        hitters_df = pd.read_csv(STANDARD_CSV)
        con = pd.read_csv(CONTRACT_CSV)
    return hitters_df, con

hitters_df, con = load_hitters_data()

print("Hitters (combined) shape:", hitters_df.shape)
print("Contracts shape:", con.shape)

# Ensure key columns exist and normalize dtypes
for df_, name in [(hitters_df, "hitters"), (con, "contracts")]:
    assert KEY_PLAYER in df_.columns, f"{name} missing column: {KEY_PLAYER}"
    if KEY_SEASON in df_.columns:
        df_[KEY_SEASON] = df_[KEY_SEASON].astype(str)
    if KEY_CONTRACT_SEASON in df_.columns:
        df_[KEY_CONTRACT_SEASON] = df_[KEY_CONTRACT_SEASON].astype(str)



Hitters (combined) shape: (665, 59)
Contracts shape: (38, 7)


In [4]:
# =====================================================================================
# 4) Prepare Hitters Data (already combined)
# =====================================================================================
hitters = hitters_df.copy()
print("Hitters shape:", hitters.shape)



Hitters shape: (665, 59)


In [5]:
# =====================================================================================
# 5) Merge Contracts
# =====================================================================================
# Prefer merging contracts on (player_name, contract_start_year) if present; otherwise (player_name, season)
contract_has_season = KEY_CONTRACT_SEASON in con.columns and con[KEY_CONTRACT_SEASON].notna().any()
merge_key = KEY_CONTRACT_SEASON if contract_has_season else KEY_SEASON

left_keys = [KEY_PLAYER, KEY_SEASON]
right_keys = [KEY_PLAYER, merge_key]

merged = hitters.merge(
    con,
    left_on=left_keys,
    right_on=right_keys,
    how="inner",  # supervised: require a known contract row
    suffixes=("", "_contract"),
)

print("Merged shape (with contracts):", merged.shape)

# Drop duplicate key from right if present
if merge_key in merged.columns and merge_key != KEY_SEASON:
    merged = merged.drop(columns=[merge_key])



Merged shape (with contracts): (93, 65)


In [6]:
# =====================================================================================
# 6) Basic Cleaning & Target Checks
# =====================================================================================
required_targets = [TARGET_YEARS_COL, TARGET_VALUE_COL]
missing_targets = merged[required_targets].isna().any(axis=1).sum()
print(f"Rows with missing targets: {missing_targets}")

df = merged.dropna(subset=required_targets).copy()

# Drop obvious non-feature columns if present
drop_if_present = ["player_id", "url_id", "team_logo_url", "notes"]
for col in drop_if_present:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Identify features/targets
y = df[required_targets].copy()
X = df.drop(columns=required_targets)

# Drop pesky "Awards" multiindex artifacts if present
for col in ["Awards", "Unnamed: 28_level_0|Awards"]:
    if col in X.columns:
        X.drop(columns=[col], inplace=True)
        print(f"Dropped column: {col}")

print("X shape:", X.shape, "| y shape:", y.shape)

# Preserve group key to avoid leakage in CV
groups = df[KEY_PLAYER].astype(str)



Rows with missing targets: 0
Dropped column: Awards
Dropped column: Unnamed: 28_level_0|Awards
X shape: (93, 60) | y shape: (93, 2)


In [6]:
# =====================================================================================
# 7) (Optional) Simple Feature Engineering
# =====================================================================================
# Example: If DOB and season exist, compute age at season.
if "dob" in df.columns:
    try:
        tmp = pd.to_datetime(df["dob"], errors="coerce")
        season_year = pd.to_numeric(df[KEY_SEASON], errors="coerce")
        X["age"] = season_year - tmp.dt.year
    except Exception as e:
        print("Age feature skipped:", e)



NameError: name 'df' is not defined

In [7]:
# =====================================================================================
# 8) Preprocessing (ColumnTransformer)
# =====================================================================================
# Recompute column groups AFTER any drops
numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
bool_cols = [c for c in X.columns if pd.api.types.is_bool_dtype(X[c])]
categorical_cols = [c for c in X.columns if c not in numeric_cols]
categorical_cols = sorted(set(categorical_cols + bool_cols))

# Exclude the grouping id from categorical encodings
exclude_cols = {KEY_PLAYER}
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

def to_str_preserve_na(Xc):
    if isinstance(Xc, pd.DataFrame):
        return Xc.where(Xc.isna(), Xc.astype(str))
    arr = np.asarray(Xc)
    mask = pd.isna(arr)
    arr = arr.astype(object)
    arr[~mask] = arr[~mask].astype(str)
    return arr

# Numeric pipeline: robust to all-NaN folds
numeric_tf = Pipeline(steps=[
    ("imp0", SimpleImputer(strategy="constant", fill_value=0)),
    ("imp1", SimpleImputer(strategy="median")),
])

# Categorical pipeline: cast→impute→OHE
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_tf = Pipeline(steps=[
    ("to_str", FunctionTransformer(to_str_preserve_na)),
    ("imp", SimpleImputer(strategy="constant", fill_value="(missing)")),
    ("ohe", ohe),
])

# ColumnTransformer (with backwards-compatible n_jobs handling)
try:
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_tf, numeric_cols),
            ("cat", categorical_tf, categorical_cols),
        ],
        remainder="drop",
        n_jobs=N_JOBS,
    )
except TypeError:
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_tf, numeric_cols),
            ("cat", categorical_tf, categorical_cols),
        ],
        remainder="drop",
    )



NameError: name 'X' is not defined

In [12]:
# =====================================================================================
# 9) Model & Pipeline
# =====================================================================================
from sklearn.preprocessing import FunctionTransformer
import numpy as np

def to_array_func(X):
    # Convert anything sparse -> dense ndarray; matrix -> ndarray; DataFrame -> ndarray
    # No-op if already an ndarray
    try:
        # SciPy sparse
        from scipy import sparse
        if sparse.issparse(X):
            return X.toarray()
    except Exception:
        pass

    # numpy.matrix -> ndarray
    if hasattr(X, "A"):  # matrix has .A alias for np.asarray
        return np.asarray(X)

    # pandas DataFrame -> ndarray
    if hasattr(X, "to_numpy"):
        return X.to_numpy()

    # Fallback
    return np.asarray(X)

to_array = FunctionTransformer(to_array_func, accept_sparse=True)

pipe = Pipeline(steps=[
    ("pre", preprocessor),
    ("to_array", to_array),
    ("model", model),
])


pipe



Pipeline(steps=[('pre',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imp0',
                                                                   SimpleImputer(fill_value=0,
                                                                                 strategy='constant')),
                                                                  ('imp1',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'WAR', 'G', 'PA',
                                                   'AB', 'R', 'H', '2B', '3B',
                                                   'HR', 'RBI', 'SB', 'CS',
                                                   'BB', 'SO', 'BA', 'OBP',
                                                   'SLG', 'OPS', 'OPS+', 'rOBA',
                                                   'Rbat+', 'TB', 'GIDP', 'HBP',
                                                   'SH', 'SF', 'IBB',
                                                   'Batti...
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Lg', 'Pos', 'Tm', 'URL_ID',
                                                   'URL_ID_contract',
                                                   'Unnamed: 27_level_0|Pos',
                                                   'pos', 'season'])])),
                ('to_array',
                 FunctionTransformer(accept_sparse=True,
                                     func=<function to_array_func at 0x17d69f1a0>)),
                ('model',
                 MultiOutputRegressor(estimator=RandomForestRegressor(n_estimators=400,
                                                                      n_jobs=-1,
                                                                      random_state=42),
                                      n_jobs=-1))])

In [8]:
# =====================================================================================
# 10) CV Splitter & Param Distributions
# =====================================================================================
# GroupKFold (to avoid same-player leakage across folds)
cv_splitter = GroupKFold(n_splits=5)

# Reasonable RF search space (applies to inner estimator via MultiOutputRegressor)
param_distributions = {
    "model__estimator__n_estimators": [200, 300, 400, 600],
    "model__estimator__max_depth": [None, 6, 10, 14, 20],
    "model__estimator__min_samples_split": [2, 5, 10],
    "model__estimator__min_samples_leaf": [1, 2, 4],
    "model__estimator__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8],
}



In [10]:
# =====================================================================================
# 11) RandomizedSearchCV (NOTE: pass groups=... here; do NOT use fit_kwargs)
# =====================================================================================
search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=cv_splitter,         # GroupKFold splitter object
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=2,
    refit=True,             # refit on full data after search
)

# Fit with groups → this is the key fix (no custom fit_kwargs dict)
search.fit(X, y, groups=groups)

print("\nBest params:")
print(json.dumps(search.best_params_, indent=2))
print("Best CV MAE (neg):", search.best_score_)

best_model = search.best_estimator_



NameError: name 'pipe' is not defined

In [9]:
# =====================================================================================
# 12) Evaluation (Holdout-style via in-sample sanity check + grouped CV scores)
# =====================================================================================
# In-sample sanity check (not a substitute for proper validation)
pred_in = best_model.predict(X)
def regression_report(y_true, y_pred, label=""):
    mae = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    cols = list(y_true.columns) if isinstance(y_true, pd.DataFrame) else [f"y{i}" for i in range(y_true.shape[1])]
    for i, c in enumerate(cols):
        print(f"[{label}] {c}: MAE={mae[i]:.3f} | RMSE={rmse[i]:.3f} | R2={r2[i]:.3f}")

regression_report(y, pred_in, label="In-sample")

# (Optional) Quick grouped CV evaluation from the search object (neg MAE)
print("\nSearch CV (neg MAE) mean:", search.cv_results_["mean_test_score"][search.best_index_])



NameError: name 'best_model' is not defined

In [17]:
# =====================================================================================
# 13) Feature Importances (Approximate)
# =====================================================================================
# --- Drop-in replacement for your get_feature_names() helper ---

from sklearn.preprocessing import OneHotEncoder

def get_feature_names_safe(preprocessor, numeric_cols, categorical_cols):
    features = list(numeric_cols)  # numeric pass-through names first

    # If there are no categorical columns, we’re done
    if not categorical_cols:
        return features

    # Try to locate the OneHotEncoder inside the 'cat' transformer
    try:
        cat = preprocessor.named_transformers_["cat"]
    except Exception as e:
        print("No 'cat' transformer found on preprocessor:", e)
        return features + list(categorical_cols)

    ohe = None
    if hasattr(cat, "named_steps"):  # it's a Pipeline
        # Try common step names
        for key in ("onehot", "ohe", "encoder", "one_hot_encoder"):
            if key in cat.named_steps:
                ohe = cat.named_steps[key]
                break
        # Or find by type
        if ohe is None:
            for _, step in cat.named_steps.items():
                if isinstance(step, OneHotEncoder):
                    ohe = step
                    break
    elif isinstance(cat, OneHotEncoder):
        ohe = cat

    if ohe is None:
        # Fallback: we can’t find the encoder—return raw categorical column names
        print("Warning: OneHotEncoder not found inside 'cat' pipeline; returning raw categorical names.")
        return features + list(categorical_cols)

    # Get OHE-expanded names
    try:
        cat_features = list(ohe.get_feature_names_out(categorical_cols))
    except TypeError:
        # Older sklearn may not accept input features
        cat_features = list(ohe.get_feature_names_out())
    except Exception as e:
        print("Failed to get categorical feature names from OHE:", e)
        cat_features = [str(c) for c in categorical_cols]

    return features + cat_features


feature_names = get_feature_names_safe(best_model.named_steps["pre"], numeric_cols, categorical_cols)

importances_per_target = []
for est in best_model.named_steps["model"].estimators_:
    try:
        importances_per_target.append(est.feature_importances_)
    except AttributeError:
        importances_per_target.append(np.zeros(len(feature_names)))

importances_mean = np.mean(importances_per_target, axis=0)
fi = pd.DataFrame({"feature": feature_names, "importance": importances_mean}).sort_values("importance", ascending=False)
fi.head(30)




,feature,importance
50,aav,0.655018
1,WAR,0.062390
10,RBI,0.022664
4,AB,0.021275
31,Batting|ISO,0.020692
3,PA,0.019601
6,H,0.015795
5,R,0.015579
32,Batting Ratios|HR%,0.013989
11,SB,0.011683


In [18]:
# =====================================================================================
# 14) Save Trained Model & Metadata
# =====================================================================================
OUTPUT_DIR = "/Users/bclem213/Desktop/BDA 602/Hitting the Market/Models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUTPUT_DIR, "hitters_contract_model.joblib")
META_PATH  = os.path.join(OUTPUT_DIR, "hitters_contract_model_meta.json")

joblib.dump(best_model, MODEL_PATH)

meta = {
    "targets": [TARGET_YEARS_COL, TARGET_VALUE_COL],
    "random_state": RANDOM_STATE,
    "feature_counts": {
        "numeric": len(numeric_cols),
        "categorical": len(categorical_cols)
    },
    "best_params": search.best_params_
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", MODEL_PATH)
print("Saved:", META_PATH)



PicklingError: Can't pickle <function to_str_preserve_na at 0x17d69f6a0>: it's not the same object as __main__.to_str_preserve_na

[CV] END model__estimator__max_depth=None, model__estimator__max_features=log2, model__estimator__min_samples_leaf=4, model__estimator__min_samples_split=10, model__estimator__n_estimators=400; total time=   0.0s
[CV] END model__estimator__max_depth=20, model__estimator__max_features=sqrt, model__estimator__min_samples_leaf=1, model__estimator__min_samples_split=10, model__estimator__n_estimators=600; total time=   0.0s
[CV] END model__estimator__max_depth=6, model__estimator__max_features=0.6, model__estimator__min_samples_leaf=1, model__estimator__min_samples_split=10, model__estimator__n_estimators=600; total time=   0.0s
[CV] END model__estimator__max_depth=None, model__estimator__max_features=0.4, model__estimator__min_samples_leaf=1, model__estimator__min_samples_split=2, model__estimator__n_estimators=200; total time=   0.0s
[CV] END model__estimator__max_depth=None, model__estimator__max_features=0.6, model__estimator__min_samples_leaf=4, model__estimator__min_samples_split=5, 

[CV] END model__estimator__max_depth=None, model__estimator__max_features=log2, model__estimator__min_samples_leaf=4, model__estimator__min_samples_split=10, model__estimator__n_estimators=400; total time=   0.0s
[CV] END model__estimator__max_depth=6, model__estimator__max_features=log2, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, model__estimator__n_estimators=600; total time=   0.0s
[CV] END model__estimator__max_depth=6, model__estimator__max_features=log2, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, model__estimator__n_estimators=600; total time=   0.0s
[CV] END model__estimator__max_depth=14, model__estimator__max_features=log2, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, model__estimator__n_estimators=200; total time=   0.0s
[CV] END model__estimator__max_depth=14, model__estimator__max_features=log2, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, mode

[CV] END model__estimator__max_depth=20, model__estimator__max_features=0.4, model__estimator__min_samples_leaf=4, model__estimator__min_samples_split=10, model__estimator__n_estimators=600; total time=   0.0s
[CV] END model__estimator__max_depth=None, model__estimator__max_features=0.4, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, model__estimator__n_estimators=400; total time=   0.0s
[CV] END model__estimator__max_depth=10, model__estimator__max_features=0.8, model__estimator__min_samples_leaf=4, model__estimator__min_samples_split=5, model__estimator__n_estimators=300; total time=   0.0s
[CV] END model__estimator__max_depth=14, model__estimator__max_features=log2, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, model__estimator__n_estimators=400; total time=   0.0s
[CV] END model__estimator__max_depth=14, model__estimator__max_features=log2, model__estimator__min_samples_leaf=2, model__estimator__min_samples_split=2, model

In [ ]:
# =====================================================================================
# 15) Inference Example
# =====================================================================================
# Take last 5 rows as "new" examples (replace with your actual inference set)
example = df.tail(5).drop(columns=[TARGET_YEARS_COL, TARGET_VALUE_COL], errors="ignore")
preds = best_model.predict(example)
pred_df = pd.DataFrame(preds, columns=[TARGET_YEARS_COL, TARGET_VALUE_COL])
inference_preview = pd.concat([example[[KEY_PLAYER, KEY_SEASON]], pred_df], axis=1)
inference_preview
